# Simple RNN with PyTorch Lightning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/simple-rnn-lightning.ipynb)

This notebook demonstrates the simplest possible Recurrent Neural Network (RNN) using PyTorch Lightning. We'll build a character-level language model that learns to predict the next character in a sequence.

**Learning Objectives:**
- Understand RNN architecture and how it processes sequences
- Learn PyTorch Lightning basics for clean, organized training
- Build a character-level language model from scratch
- Visualize how RNNs maintain hidden state across time steps

## Part 1: Setup and Dependencies

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import lightning as L
import matplotlib.pyplot as plt
import numpy as np

# Set random seed for reproducibility
L.seed_everything(42)

# Check device
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

## Part 2: Understanding RNNs

A Recurrent Neural Network processes sequences one element at a time, maintaining a **hidden state** that carries information from previous time steps.

**Key equation:**
```
h_t = tanh(W_ih * x_t + W_hh * h_{t-1} + b)
```

Where:
- `x_t` = input at time step t
- `h_t` = hidden state at time step t
- `W_ih` = input-to-hidden weights
- `W_hh` = hidden-to-hidden weights (this creates the recurrence!)
- `b` = bias

**Visual representation:**
```
Input:  x_0 → x_1 → x_2 → x_3
         ↓     ↓     ↓     ↓
Hidden: h_0 → h_1 → h_2 → h_3
         ↓     ↓     ↓     ↓
Output: y_0   y_1   y_2   y_3
```

## Part 3: Prepare Simple Text Data

We'll use a tiny dataset - just a few simple sentences. This makes it easy to understand what the model is learning.

In [ ]:
# Simple training text
text = """hello world
hello there
world peace
deep learning
neural networks
recurrent neural networks
pytorch lightning
"""

# Create character vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {chars}")

# Create mappings
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

# Encode the entire text
encoded_text = [char_to_idx[ch] for ch in text]
print(f"\nText length: {len(text)} characters")
print(f"Encoded text (first 50): {encoded_text[:50]}")

## Part 4: Create Dataset

We'll create sequences of characters where:
- Input: sequence of N characters
- Target: the next character after each input character

For example:
- Input: "hell" → Targets: "ello"

In [ ]:
class CharDataset(Dataset):
    def __init__(self, text, char_to_idx, seq_length=10):
        self.text = text
        self.char_to_idx = char_to_idx
        self.seq_length = seq_length
        
        # Encode text
        self.encoded = [char_to_idx[ch] for ch in text]
    
    def __len__(self):
        # Number of sequences we can create
        return len(self.encoded) - self.seq_length
    
    def __getitem__(self, idx):
        # Get sequence of characters
        x = self.encoded[idx:idx + self.seq_length]
        # Target is the next character after each input character
        y = self.encoded[idx + 1:idx + self.seq_length + 1]
        
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

# Create dataset
seq_length = 10
dataset = CharDataset(text, char_to_idx, seq_length)

print(f"Dataset size: {len(dataset)} sequences")

# Show example
x_sample, y_sample = dataset[0]
print(f"\nExample sequence:")
print(f"Input chars:  {''.join([idx_to_char[i.item()] for i in x_sample])}")
print(f"Target chars: {''.join([idx_to_char[i.item()] for i in y_sample])}")
print(f"Input indices:  {x_sample.tolist()}")
print(f"Target indices: {y_sample.tolist()}")

## Part 5: Build RNN Model with Lightning

PyTorch Lightning organizes our code into clear sections:
- `__init__`: Define model architecture
- `forward`: How data flows through the model
- `training_step`: What happens during training
- `configure_optimizers`: How we optimize the model

In [ ]:
class SimpleRNN(L.LightningModule):
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64, learning_rate=0.001):
        super().__init__()
        self.save_hyperparameters()
        
        # Embedding layer: converts character indices to dense vectors
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # RNN layer: processes sequences and maintains hidden state
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True  # Input shape: (batch, seq, features)
        )
        
        # Output layer: converts hidden state to vocabulary predictions
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, hidden=None):
        # x shape: (batch, seq_length)
        
        # 1. Embed characters
        embedded = self.embedding(x)  # (batch, seq_length, embedding_dim)
        
        # 2. Process through RNN
        output, hidden = self.rnn(embedded, hidden)  # output: (batch, seq_length, hidden_dim)
        
        # 3. Convert to character predictions
        logits = self.fc(output)  # (batch, seq_length, vocab_size)
        
        return logits, hidden
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        
        # Forward pass
        logits, _ = self(x)
        
        # Reshape for cross entropy loss
        # logits: (batch, seq_length, vocab_size) → (batch * seq_length, vocab_size)
        # targets: (batch, seq_length) → (batch * seq_length)
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1)
        )
        
        # Log metrics
        self.log('train_loss', loss, prog_bar=True)
        return loss
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

# Create model
model = SimpleRNN(vocab_size=vocab_size, embedding_dim=32, hidden_dim=64, learning_rate=0.001)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

## Part 6: Train the Model

We'll train for a small number of epochs. With such a tiny dataset, the model should learn quickly.

In [ ]:
# Create data loader
train_loader = DataLoader(dataset, batch_size=8, shuffle=True)

# Create trainer
trainer = L.Trainer(
    max_epochs=100,
    accelerator='auto',  # Automatically uses GPU if available
    devices=1,
    enable_progress_bar=True,
    log_every_n_steps=5
)

# Train
trainer.fit(model, train_loader)

## Part 7: Generate Text

Now let's use our trained RNN to generate new text! We'll:
1. Start with a seed character
2. Feed it through the RNN to get predictions
3. Sample the next character from the predictions
4. Use that character as the next input
5. Repeat!

In [ ]:
def generate_text(model, start_char, length=100, temperature=1.0):
    """
    Generate text using the trained RNN.
    
    Args:
        model: Trained RNN model
        start_char: Character to start generation
        length: Number of characters to generate
        temperature: Sampling temperature (higher = more random, lower = more conservative)
    """
    model.eval()
    
    # Start with the seed character
    current_char = start_char
    generated = current_char
    
    # Hidden state starts as None
    hidden = None
    
    with torch.no_grad():
        for _ in range(length):
            # Encode current character
            x = torch.tensor([[char_to_idx[current_char]]], dtype=torch.long)
            
            # Get predictions
            logits, hidden = model(x, hidden)
            
            # Apply temperature and get probabilities
            probs = F.softmax(logits[0, -1] / temperature, dim=0)
            
            # Sample next character
            next_idx = torch.multinomial(probs, 1).item()
            current_char = idx_to_char[next_idx]
            
            generated += current_char
    
    return generated

# Generate text with different starting characters and temperatures
print("Generated text samples:\n")

for start_char in ['h', 'w', 'd']:
    print(f"Starting with '{start_char}':")
    
    # Try different temperatures
    for temp in [0.5, 1.0, 1.5]:
        generated = generate_text(model, start_char, length=50, temperature=temp)
        print(f"  Temperature {temp}: {repr(generated)}")
    print()

## Part 8: Visualize Hidden States

Let's peek inside the RNN to see how the hidden state changes as it processes a sequence.

In [ ]:
def visualize_hidden_states(model, text_input):
    """
    Visualize how the RNN's hidden state evolves as it processes text.
    """
    model.eval()
    
    # Encode input
    encoded = [char_to_idx[ch] for ch in text_input]
    x = torch.tensor([encoded], dtype=torch.long)
    
    # Get hidden states at each time step
    with torch.no_grad():
        embedded = model.embedding(x)
        output, hidden = model.rnn(embedded)
        
        # output shape: (1, seq_length, hidden_dim)
        hidden_states = output[0].cpu().numpy()  # (seq_length, hidden_dim)
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    
    # Plot 1: Heatmap of hidden states
    im = ax1.imshow(hidden_states.T, aspect='auto', cmap='coolwarm', interpolation='nearest')
    ax1.set_xlabel('Time Step (Character Position)')
    ax1.set_ylabel('Hidden Dimension')
    ax1.set_title('RNN Hidden State Evolution')
    ax1.set_xticks(range(len(text_input)))
    ax1.set_xticklabels(list(text_input))
    plt.colorbar(im, ax=ax1, label='Activation Value')
    
    # Plot 2: Track a few specific hidden dimensions over time
    for i in range(min(5, hidden_states.shape[1])):
        ax2.plot(hidden_states[:, i], label=f'Hidden dim {i}', marker='o', markersize=4)
    
    ax2.set_xlabel('Time Step (Character Position)')
    ax2.set_ylabel('Activation Value')
    ax2.set_title('Selected Hidden Dimensions Over Time')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_xticks(range(len(text_input)))
    ax2.set_xticklabels(list(text_input))
    
    plt.tight_layout()
    plt.show()

# Visualize a sample phrase
sample_text = "hello world"
print(f"Visualizing hidden states for: '{sample_text}'\n")
visualize_hidden_states(model, sample_text)

## Part 9: Understanding the Visualizations

**What do these plots tell us?**

1. **Heatmap (top)**: Shows all 64 hidden dimensions over time. Each column is one character.
   - Red = positive activation
   - Blue = negative activation
   - Notice how patterns change as different characters are processed

2. **Line plot (bottom)**: Tracks specific hidden dimensions.
   - See how values evolve as the sequence progresses
   - Some dimensions may respond to specific patterns (e.g., spaces, vowels)

**Key insight**: The hidden state is the RNN's "memory" - it encodes information about what it has seen so far in the sequence.

## Part 10: Predict Next Character (Interactive)

Let's create an interactive function that shows what the model thinks should come next after any input text.

In [ ]:
def predict_next_char(model, text_input, top_k=5):
    """
    Show the top-k most likely next characters after the input text.
    """
    model.eval()
    
    # Encode input
    encoded = [char_to_idx[ch] for ch in text_input]
    x = torch.tensor([encoded], dtype=torch.long)
    
    # Get predictions
    with torch.no_grad():
        logits, _ = model(x)
        # Get predictions for the last character
        probs = F.softmax(logits[0, -1], dim=0)
    
    # Get top-k predictions
    top_probs, top_indices = torch.topk(probs, k=min(top_k, vocab_size))
    
    print(f"Input: '{text_input}'")
    print(f"\nTop {top_k} predictions for next character:")
    print("-" * 40)
    
    for i, (prob, idx) in enumerate(zip(top_probs, top_indices), 1):
        char = idx_to_char[idx.item()]
        char_display = repr(char) if char in ['\n', ' ', '\t'] else char
        print(f"{i}. {char_display:6} : {prob.item()*100:5.2f}%")
    
    # Visualize probabilities
    plt.figure(figsize=(12, 4))
    plt.bar(range(vocab_size), probs.cpu().numpy())
    plt.xlabel('Character Index')
    plt.ylabel('Probability')
    plt.title(f"Next Character Probability Distribution after '{text_input}'")
    
    # Annotate top predictions
    for idx in top_indices[:3]:
        plt.text(idx, probs[idx].item() + 0.01, idx_to_char[idx.item()], 
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

# Try different inputs
test_inputs = ['hell', 'worl', 'deep ', 'neur']

for test_input in test_inputs:
    predict_next_char(model, test_input, top_k=5)
    print("\n" + "="*60 + "\n")

## Part 11: Key Takeaways

**What we learned:**

1. **RNN Architecture**
   - RNNs process sequences one element at a time
   - Hidden state carries information across time steps
   - Three components: embedding → RNN → output layer

2. **PyTorch Lightning Benefits**
   - Clean separation of model architecture and training logic
   - Automatic device handling (CPU/GPU/MPS)
   - Built-in logging and progress tracking
   - Less boilerplate code

3. **Character-Level Language Modeling**
   - Model learns patterns in character sequences
   - Can generate new text by sampling predictions
   - Temperature controls randomness in generation

4. **RNN Limitations** (not covered here, but important):
   - Vanishing/exploding gradients for long sequences
   - Difficulty learning long-term dependencies
   - Solutions: LSTM, GRU, Transformers

## Part 12: Experiments to Try

**Challenge yourself:**

1. **Bigger dataset**: Use a larger text corpus (e.g., a book from Project Gutenberg)
2. **Deeper RNN**: Try `num_layers=2` or `num_layers=3` in the RNN
3. **Different architecture**: Replace `nn.RNN` with `nn.LSTM` or `nn.GRU`
4. **Tune hyperparameters**: Experiment with:
   - Hidden dimension size
   - Embedding dimension
   - Learning rate
   - Sequence length
5. **Word-level model**: Instead of characters, predict the next word
6. **Sentiment analysis**: Use RNN to classify text sentiment (requires labeled dataset)

**Questions to explore:**
- How does sequence length affect training and generation quality?
- What happens with a much larger hidden dimension?
- Can you visualize attention-like patterns in the hidden states?